In [ ]:
import glob
import numpy as np
import polars as pl

In [ ]:
# Get gene trait associations
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/regenie_127phenotypes_lofteeHC_mac20_EUR.parquet -o /home/dnanexus/data_dir/
# !dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/loftee_mac20_associations_bh_corrected.parquet -o /home/dnanexus/data_dir/

gene_trait_df = (
    pl.read_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR.parquet')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)


gene_trait_df

Error: path
"/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR.parquet"
already exists but -f/--overwrite was not set
Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_mac20_lofteeHC_EUR_
correlations.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [ ]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

Error: path "/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv"
already exists but -f/--overwrite was not set


['1000020', '1000107', '1000161', '1000172', '1000221']

In [ ]:
# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)


Error: path "/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet"
already exists but -f/--overwrite was not set
shape: (121, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u64    │
╞═════════════════════════════════╪════════╡
│ townsend_deprivation_index_at_… ┆ 378461 │
│ waist_circumference_int         ┆ 378281 │
│ hip_circumference_int           ┆ 378244 │
│ standing_height_int             ┆ 378108 │
│ weight_int                      ┆ 377841 │
│ …                               ┆ …      │
│ heel_bone_mineral_density_bmd_… ┆ 118537 │
│ heel_bone_mineral_density_bmd_… ┆ 118535 │
│ microalbumin_in_urine_int       ┆ 114444 │
│ age_at_hysterectomy_int         ┆ 38948  │
│ stroke_volume_during_pwa_int    ┆ 30652  │
└─────────────────────────────────┴────────┘


sample,phenotype,pheno_value
str,str,f64
"""1000020""","""hand_grip_strength_left_int""",-0.440416
"""1000107""","""hand_grip_strength_left_int""",0.471972
"""1000161""","""hand_grip_strength_left_int""",0.100091
"""1000172""","""hand_grip_strength_left_int""",-1.197228
"""1000221""","""hand_grip_strength_left_int""",0.417089
…,…,…
"""4974782""","""mean_corpuscular_haemoglobin_i…",-1.652449
"""5956310""","""mean_corpuscular_haemoglobin_i…",0.604435
"""4301443""","""mean_corpuscular_haemoglobin_i…",0.051969


In [ ]:
# Get null counts and transpose for easier viewing
null_counts = phenos.null_count()

# Convert to a format that's easier to read (column name -> count)
null_counts_long = null_counts.transpose(include_header=True, column_names=['null_count'])

less_miss = null_counts_long.filter(pl.col('null_count') <= 0.2*len(unrel_eur_samples)).sort('null_count', descending=True)

traits2keep = gene_trait_df.filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))['phenotype'].unique().to_list()

print(len(traits2keep))

gene_trait_df.filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))

102


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [ ]:
(
    pl.read_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR.parquet')
    .filter(pl.col('phenotype').is_in(less_miss['column'].to_list()))
    .write_parquet('/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet')
)

!dx upload /home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results/
#1_683_399

[===========================================================>] Uploaded 96,552,882 of 96,552,882 bytes (100%) /home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet
ID                                file-J669xv0Jg0yJFFvvx2p989YQ
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/REGENIE_results
Name                              regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Wed Feb 11 15:23:28 2026
Created by                        shubhankar
 via the job                      job-J669QQ0Jg0yFfBqKq47Z2fqB
Last modified                     Wed Feb 11 15:23:30 2026
Media type              